In [0]:
from pyspark.sql import functions as F

# Path where the file is content
file_path = "/Workspace/Users/juanes.pelaez18@gmail.com/vehicle_position_20251111_192042.json"

# Simple batch read of JSON file of vehicles positions
df = (spark.read
      .option("multiLine", True)   
      .option("mode", "PERMISSIVE")
      .json(file_path))

# Show the spark DataFrame
df.show()


+--------------------+--------------------+
|                  id|             vehicle|
+--------------------+--------------------+
|vehicle_position_...|{INCOMING_AT, NUL...|
|vehicle_position_...|{IN_TRANSIT_TO, N...|
|vehicle_position_...|{IN_TRANSIT_TO, N...|
|vehicle_position_...|{STOPPED_AT, NULL...|
|vehicle_position_...|{IN_TRANSIT_TO, N...|
|vehicle_position_...|{STOPPED_AT, NULL...|
|vehicle_position_...|{STOPPED_AT, NULL...|
|vehicle_position_...|{STOPPED_AT, NULL...|
|vehicle_position_...|{INCOMING_AT, NUL...|
|vehicle_position_...|{IN_TRANSIT_TO, N...|
|vehicle_position_...|{STOPPED_AT, NULL...|
|vehicle_position_...|{INCOMING_AT, NUL...|
|vehicle_position_...|{IN_TRANSIT_TO, N...|
|vehicle_position_...|{IN_TRANSIT_TO, N...|
|vehicle_position_...|{IN_TRANSIT_TO, N...|
|vehicle_position_...|{STOPPED_AT, NULL...|
|vehicle_position_...|{IN_TRANSIT_TO, N...|
|vehicle_position_...|{INCOMING_AT, NUL...|
|vehicle_position_...|{IN_TRANSIT_TO, N...|
|vehicle_position_...|{STOPPED_A

Let\'s examine the structure of the Spark DataFrame that we created from the Pandas DataFrame. Understanding the schema of a DataFrame is crucial as it provides insight into the data types of each column and helps ensure that the data is organized correctly for analysis.

In [0]:
print("Schema of the Spark DataFrame:")
df.printSchema()
# Print the structure of the DataFrame (columns and types)

Schema of the Spark DataFrame:
root
 |-- id: string (nullable = true)
 |-- vehicle: struct (nullable = true)
 |    |-- current_status: string (nullable = true)
 |    |-- occupancy_status: string (nullable = true)
 |    |-- position: struct (nullable = true)
 |    |    |-- bearing: double (nullable = true)
 |    |    |-- latitude: double (nullable = true)
 |    |    |-- longitude: double (nullable = true)
 |    |    |-- odometer: double (nullable = true)
 |    |    |-- speed: double (nullable = true)
 |    |-- stop_id: string (nullable = true)
 |    |-- timestamp: string (nullable = true)
 |    |-- trip: struct (nullable = true)
 |    |    |-- direction_id: long (nullable = true)
 |    |    |-- route_id: string (nullable = true)
 |    |    |-- schedule_relationship: string (nullable = true)
 |    |    |-- start_date: string (nullable = true)
 |    |    |-- start_time: string (nullable = true)
 |    |-- vehicle: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |

let\'s perform basic data exploration on the Spark DataFrame. This step is essential for understanding the data set better, allowing us to gain insights and identify any patterns or anomalies. For this case lets check vehicle column, since it is an struct that contain multiple information

In [0]:
columns_to_display = ['vehicle.current_status', 'vehicle.occupancy_status', 'vehicle.position.latitude', 'vehicle.position.longitude', 'vehicle.position.speed', 'vehicle.stop_id', 'vehicle.trip.route_id', 'vehicle.trip.direction_id','vehicle.vehicle.id']
# Display the first 5 records of the specified columns
df.select(columns_to_display).show(10)

+--------------+----------------+---------+---------+-----+-------+--------+------------+------+
|current_status|occupancy_status| latitude|longitude|speed|stop_id|route_id|direction_id|    id|
+--------------+----------------+---------+---------+-----+-------+--------+------------+------+
|   INCOMING_AT|            NULL| 60.16705|24.636606| 5.17|2421202|    2158|           0|22/943|
| IN_TRANSIT_TO|            NULL|60.185226|24.957222|17.72|1121601|    31M1|           0|50/165|
| IN_TRANSIT_TO|            NULL| 60.20961|25.076067|16.19|1453601|    31M1|           0|50/151|
|    STOPPED_AT|            NULL|60.175926| 24.82954|11.85|2222601|    31M2|           0|50/155|
| IN_TRANSIT_TO|            NULL|60.262592| 25.26476|15.86|9204202|    9841|           0|18/148|
|    STOPPED_AT|            NULL|60.189034|25.009773|14.74|1420602|    31M2|           1|50/181|
|    STOPPED_AT|            NULL| 60.29821| 25.32903| 0.01|9208214|    9841|           1|18/155|
|    STOPPED_AT|            NU

In [0]:
df_clean = (
    df
    .withColumn("vehicle_id", F.col("vehicle.vehicle.id"))
    .withColumn("route_id", F.col("vehicle.trip.route_id"))
    .withColumn("trip_start_time", F.col("vehicle.trip.start_time"))
    .withColumn("lat", F.col("vehicle.position.latitude"))
    .withColumn("lon", F.col("vehicle.position.longitude"))
    .withColumn("speed", F.col("vehicle.position.speed"))
    .withColumn("ts_raw", F.col("vehicle.timestamp"))
)

df_clean.select(
    "vehicle_id", 
    "route_id",
    "trip_start_time",
    "lat", 
    "lon", 
    "speed",
    "ts_raw"
).show(10, truncate=False)


+----------+--------+---------------+---------+---------+-----+----------+
|vehicle_id|route_id|trip_start_time|lat      |lon      |speed|ts_raw    |
+----------+--------+---------------+---------+---------+-----+----------+
|22/943    |2158    |20:59:00       |60.16705 |24.636606|5.17 |1762888841|
|50/165    |31M1    |20:46:00       |60.185226|24.957222|17.72|1762888841|
|50/151    |31M1    |20:36:00       |60.20961 |25.076067|16.19|1762888841|
|50/155    |31M2    |21:16:00       |60.175926|24.82954 |11.85|1762888841|
|18/148    |9841    |21:00:00       |60.262592|25.26476 |15.86|1762888841|
|50/181    |31M2    |21:08:00       |60.189034|25.009773|14.74|1762888841|
|18/155    |9841    |21:05:00       |60.29821 |25.32903 |0.01 |1762888841|
|18/156    |9844    |21:20:00       |60.21027 |25.07713 |2.62 |1762888841|
|18/157    |1831K   |21:05:00       |60.251114|25.172619|17.27|1762888841|
|50/169    |31M1    |21:06:00       |60.174084|24.796675|21.17|1762888841|
+----------+--------+----

Now lets transformt the timestap to a proper format in order to be able to use it as a timestamp column

In [0]:
df_clean = df_clean.withColumn('event_ts',F.to_timestamp(F.col('ts_raw').cast("long")))


df_clean.select(
    "vehicle_id", 
    "route_id",
    "trip_start_time",
    "lat", 
    "lon", 
    "speed",
    "event_ts"
).show(10, truncate=False)

+----------+--------+---------------+---------+---------+-----+-------------------+
|vehicle_id|route_id|trip_start_time|lat      |lon      |speed|event_ts           |
+----------+--------+---------------+---------+---------+-----+-------------------+
|22/943    |2158    |20:59:00       |60.16705 |24.636606|5.17 |2025-11-11 19:20:41|
|50/165    |31M1    |20:46:00       |60.185226|24.957222|17.72|2025-11-11 19:20:41|
|50/151    |31M1    |20:36:00       |60.20961 |25.076067|16.19|2025-11-11 19:20:41|
|50/155    |31M2    |21:16:00       |60.175926|24.82954 |11.85|2025-11-11 19:20:41|
|18/148    |9841    |21:00:00       |60.262592|25.26476 |15.86|2025-11-11 19:20:41|
|50/181    |31M2    |21:08:00       |60.189034|25.009773|14.74|2025-11-11 19:20:41|
|18/155    |9841    |21:05:00       |60.29821 |25.32903 |0.01 |2025-11-11 19:20:41|
|18/156    |9844    |21:20:00       |60.21027 |25.07713 |2.62 |2025-11-11 19:20:41|
|18/157    |1831K   |21:05:00       |60.251114|25.172619|17.27|2025-11-11 19

In [0]:
df_ready = (
    df_clean
    .filter(F.col("lat").between(-90, 90) & F.col("lon").between(-180, 180))
    .dropna(subset=["vehicle_id", "event_ts"])
    .dropDuplicates(["vehicle_id", "event_ts"])
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn("event_hour", F.date_format("event_ts", "HH:MM"))
)

df_ready.select(
    "vehicle_id", "route_id", "lat", "lon", "speed", "event_ts", "event_date", "event_hour"
).show(10, truncate=False)


+----------+--------+---------+---------+-----+-------------------+----------+----------+
|vehicle_id|route_id|lat      |lon      |speed|event_ts           |event_date|event_hour|
+----------+--------+---------+---------+-----+-------------------+----------+----------+
|18/432    |2542    |60.13501 |24.669216|1.97 |2025-11-11 19:20:41|2025-11-11|19:11     |
|40/470    |1003    |60.17517 |24.950426|7.77 |2025-11-11 19:20:41|2025-11-11|19:11     |
|12/1824   |4400    |60.26056 |24.877148|0.57 |2025-11-11 19:20:41|2025-11-11|19:11     |
|22/1394   |2213N   |60.191593|24.899506|11.41|2025-11-11 19:20:41|2025-11-11|19:11     |
|90/1020   |3001K   |60.22842 |24.966963|0.0  |2025-11-11 19:20:41|2025-11-11|19:11     |
|18/1014   |1500    |60.20926 |25.056875|4.6  |2025-11-11 19:20:41|2025-11-11|19:11     |
|18/1081   |1065    |60.216866|24.959406|0.0  |2025-11-11 19:20:41|2025-11-11|19:11     |
|40/613    |2015    |60.23101 |24.98106 |14.64|2025-11-11 19:20:41|2025-11-11|19:11     |
|12/2503  

Convert to Delta

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS hsl_demo")

(
    df_ready
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("event_date", "event_hour")
    .saveAsTable("hsl_demo.vehicle_positions_silver")
)

df_check = spark.table("hsl_demo.vehicle_positions_silver")
display(df_check)
print(df_check.count())



id,vehicle,vehicle_id,route_id,trip_start_time,lat,lon,speed,ts_raw,event_ts,event_date,event_hour
vehicle_position_18/432,"List(STOPPED_AT, null, List(229.0, 60.13501, 24.669216, 16667.0, 1.97), 2412237, 1762888841, List(1, 2542, SCHEDULED, 20251111, 20:39:00), List(18/432, null))",18/432,2542,20:39:00,60.13501,24.669216,1.97,1762888841,2025-11-11T19:20:41.000Z,2025-11-11,19:11
vehicle_position_40/470,"List(IN_TRANSIT_TO, null, List(356.0, 60.17517, 24.950426, 3343.0, 7.77), 1111428, 1762888841, List(0, 1003, SCHEDULED, 20251111, 21:01:00), List(40/470, null))",40/470,1003,21:01:00,60.17517,24.950426,7.77,1762888841,2025-11-11T19:20:41.000Z,2025-11-11,19:11
vehicle_position_12/1824,"List(STOPPED_AT, CRUSHED_STANDING_ROOM_ONLY, List(345.0, 60.26056, 24.877148, 11462.0, 0.57), 1334128, 1762888841, List(0, 4400, SCHEDULED, 20251111, 20:51:00), List(12/1824, null))",12/1824,4400,20:51:00,60.26056,24.877148,0.57,1762888841,2025-11-11T19:20:41.000Z,2025-11-11,19:11
vehicle_position_22/1394,"List(IN_TRANSIT_TO, FEW_SEATS_AVAILABLE, List(302.0, 60.191593, 24.899506, 3577.0, 11.41), 1150111, 1762888841, List(0, 2213N, SCHEDULED, 20251111, 21:08:00), List(22/1394, null))",22/1394,2213N,21:08:00,60.191593,24.899506,11.41,1762888841,2025-11-11T19:20:41.000Z,2025-11-11,19:11
vehicle_position_90/1020,"List(STOPPED_AT, null, List(39.0, 60.22842, 24.966963, 6891.0, 0.0), 1285501, 1762888841, List(0, 3001K, SCHEDULED, 20251111, 21:11:00), List(90/1020, null))",90/1020,3001K,21:11:00,60.22842,24.966963,0.0,1762888841,2025-11-11T19:20:41.000Z,2025-11-11,19:11
vehicle_position_18/1014,"List(STOPPED_AT, null, List(210.0, 60.20926, 25.056875, 1325.0, 4.6), 1456153, 1762888841, List(0, 1500, SCHEDULED, 20251111, 21:17:00), List(18/1014, null))",18/1014,1500,21:17:00,60.20926,25.056875,4.6,1762888841,2025-11-11T19:20:41.000Z,2025-11-11,19:11
vehicle_position_18/1081,"List(STOPPED_AT, null, List(209.0, 60.216866, 24.959406, 2351.0, 0.0), 1250180, 1762888841, List(1, 1065, SCHEDULED, 20251111, 21:16:00), List(18/1081, null))",18/1081,1065,21:16:00,60.216866,24.959406,0.0,1762888841,2025-11-11T19:20:41.000Z,2025-11-11,19:11
vehicle_position_40/613,"List(IN_TRANSIT_TO, FEW_SEATS_AVAILABLE, List(94.0, 60.23101, 24.98106, 17447.0, 14.64), 1364402, 1762888841, List(1, 2015, SCHEDULED, 20251111, 20:35:00), List(40/613, null))",40/613,2015,20:35:00,60.23101,24.98106,14.64,1762888841,2025-11-11T19:20:41.000Z,2025-11-11,19:11
vehicle_position_12/2503,"List(IN_TRANSIT_TO, MANY_SEATS_AVAILABLE, List(29.0, 60.23774, 24.97595, 11457.0, 11.69), 1370103, 1762888841, List(1, 2553, SCHEDULED, 20251111, 20:57:00), List(12/2503, null))",12/2503,2553,20:57:00,60.23774,24.97595,11.69,1762888841,2025-11-11T19:20:41.000Z,2025-11-11,19:11
vehicle_position_22/1438,"List(IN_TRANSIT_TO, MANY_SEATS_AVAILABLE, List(241.0, 60.237453, 24.880348, 10850.0, 7.96), 1331166, 1762888841, List(0, 1056, SCHEDULED, 20251111, 20:47:00), List(22/1438, null))",22/1438,1056,20:47:00,60.237453,24.880348,7.96,1762888841,2025-11-11T19:20:41.000Z,2025-11-11,19:11


604
